### Libs

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import os
import seaborn as sns
import functions as fn

### Aply

In [2]:
lista_csv, nomes = fn.get_file_paths(r'C:\Users\tiago\UFSC (local)\Prologis\Panic buyng\Ambiental clima\Dados clima')
lista_csv

['C:\\Users\\tiago\\UFSC (local)\\Prologis\\Panic buyng\\Ambiental clima\\Dados clima\\dados_A806_H_2018-01-01_2025-12-31.csv',
 'C:\\Users\\tiago\\UFSC (local)\\Prologis\\Panic buyng\\Ambiental clima\\Dados clima\\dados_A814_H_2018-01-01_2025-12-31.csv',
 'C:\\Users\\tiago\\UFSC (local)\\Prologis\\Panic buyng\\Ambiental clima\\Dados clima\\dados_A815_H_2018-01-01_2025-12-31.csv',
 'C:\\Users\\tiago\\UFSC (local)\\Prologis\\Panic buyng\\Ambiental clima\\Dados clima\\dados_A816_H_2018-01-01_2025-12-31.csv',
 'C:\\Users\\tiago\\UFSC (local)\\Prologis\\Panic buyng\\Ambiental clima\\Dados clima\\dados_A817_H_2018-01-01_2025-12-31.csv',
 'C:\\Users\\tiago\\UFSC (local)\\Prologis\\Panic buyng\\Ambiental clima\\Dados clima\\dados_A841_H_2018-01-01_2025-12-31.csv',
 'C:\\Users\\tiago\\UFSC (local)\\Prologis\\Panic buyng\\Ambiental clima\\Dados clima\\dados_A845_H_2018-01-01_2025-12-31.csv',
 'C:\\Users\\tiago\\UFSC (local)\\Prologis\\Panic buyng\\Ambiental clima\\Dados clima\\dados_A848_H_2018

In [3]:

# 2. Agrupamento: Criar um dicionário onde a chave é a cidade e o valor é a lista de seus arquivos
files_by_city = {}
for path, nome in zip(lista_csv, nomes):
    if nome not in files_by_city:
        files_by_city[nome] = []
    files_by_city[nome].append(path)

dbs = {}

# Agrupamos os arquivos por cidade para processar um de cada vez
for nome, city_files in files_by_city.items():
    print(f"Processando dados de: {nome}")
    
    # Se você tem vários arquivos por cidade, aqui estamos pegando o primeiro [0]
    # Se quiser todos, precisaríamos de um pequeno ajuste para concatenar os arquivos
    path_do_arquivo = city_files[0] 
    
    # Chamada da sua função exatamente como você a definiu
    db, est, info, dbd, dbm = fn.data_analisys(nome, path_do_arquivo)
    
    # Guardando na estrutura de Dicionário de Dicionários
    dbs[nome] = {
        "dataframe": db,
        "estatisticas": est,
        "info": info,
        "df_days": dbd,
        "df_months": dbm
    }

print("Processamento concluído com sucesso!")

Processando dados de: FLORIANOPOLIS
Processando dados de: URUSSANGA
Processando dados de: SAO JOAQUIM
Processando dados de: NOVO HORIZONTE
Processando dados de: INDAIAL
Processando dados de: JOACABA
Processando dados de: BOM JARDIM DA SERRA - MORRO DA IGREJA
Processando dados de: DIONISIO CERQUEIRA
Processando dados de: ITAPOA
Processando dados de: SAO MIGUEL DO OESTE
Processando dados de: XANXERE
Processando dados de: CACADOR
Processando dados de: CURITIBANOS
Processando dados de: RIO DO CAMPO
Processando dados de: RIO NEGRINHO
Processando dados de: ITUPORANGA
Processando dados de: MAJOR VIEIRA
Processando dados de: LAGES
Processando dados de: Laguna - Farol de Santa Marta
Processando dados de: ARARANGUA
Processando dados de: ITAJAI
Processando dados de: RANCHO QUEIMADO
Processando dados de: CHAPECO
Processando dados de: CAMPOS NOVOS
Processando dados de: CURITIBANOS - EPAGRI
Processamento concluído com sucesso!


### Gráficos e tabelas

In [4]:
#Visualização recipitação mensal comparativa 
#Tabela acumulada 3 dias por cidade

periodo = ['Data Medicao']
df_cities = pd.DataFrame()
cols = []

dfs_mes = []
dfs_diario = []
info_total = pd.DataFrame()

for nome in nomes:
    df = dbs[nome]['df_days']
    dfs_diario.append(df)
    dias_criticos = df[df['PrecAc_3dias'] >= 100]
    dbs[nome]['dias_criticos'] = dias_criticos

    info_total = pd.concat([info_total, dbs[nome]['info']])

    dfs_mes.append(dbs[nome]['df_months'])

    title = 'pac_'+ f'{nome}'
    df = dbs[nome]['df_days']
    df = df.set_index('Data Medicao')


    col = df[['PrecAc_3dias']].copy()
    cols.append(col.rename(columns={'PrecAc_3dias': title}))


df_cities = pd.concat(cols, axis=1)
df_cities.reset_index(inplace=True)

# Salvar múltiplos DataFrames em um único arquivo Excel
nome_arquivo = 'relatorio_chuvas.xlsx'

with pd.ExcelWriter(nome_arquivo, engine='openpyxl') as writer:
    # 1. Primeira planilha: info_total
    info_total.to_excel(writer, sheet_name='Informações', index=False)
    
    # 2. Segunda planilha: df_cities
    df_cities.to_excel(writer, sheet_name='Comparativo Cidades', index=False)
    
    # 3. Subsequentes: dias críticos de cada cidade
    for nome in nomes:
        df_criticos = dbs[nome].get('dias_criticos')
        sheet_name = f'Crit_{nome}'[:31] 
        df_criticos.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Arquivo '{nome_arquivo}' gerado com sucesso!")

Arquivo 'relatorio_chuvas.xlsx' gerado com sucesso!


In [6]:
#plots
fn.plot_Prec_multi(dfs_mes, nomes, 'AnoMes', 'PRECIPITACAO TOTAL, HORARIO(mm)', 'Precipitação mensal')
fn.plot_Prec_multi(dfs_diario, nomes, 'Data Medicao', 'PRECIPITACAO TOTAL, HORARIO(mm)', 'Precipitação diaria')
fn.plot_Prec_multi(dfs_diario, nomes, 'Data Medicao', 'PrecAc_3dias', 'Precipitação acumulada diaria')

In [5]:
#informações gerais
info_total

,nome,qtd_registros,qtd_nulos_chuva,Relacao_nulos,precip_total
0,FLORIANOPOLIS,70128,8018,11.43%,12645.6
0,URUSSANGA,70128,13107,18.69%,11554.2
0,SAO JOAQUIM,70128,3786,5.40%,13471.8
0,NOVO HORIZONTE,70128,27439,39.13%,9341.0
0,INDAIAL,70128,19686,28.07%,10427.8
0,JOACABA,70128,14049,20.03%,12379.8
0,BOM JARDIM DA SERRA - MORRO DA IGREJA,70128,22774,32.47%,16134.0
0,DIONISIO CERQUEIRA,70128,3926,5.60%,16328.4
0,ITAPOA,70128,12571,17.93%,14953.0
0,SAO MIGUEL DO OESTE,70128,32827,46.81%,7728.2


In [7]:
#Tabela acumulada 3 dias por cidade
df_cities

,Data Medicao,pac_FLORIANOPOLIS,pac_URUSSANGA,pac_DIONISIO CERQUEIRA,pac_SAO MIGUEL DO OESTE,pac_XANXERE,pac_CACADOR,pac_CURITIBANOS,pac_RIO DO CAMPO,pac_ITUPORANGA,pac_LAGES,pac_ARARANGUA,pac_ITAJAI,pac_RANCHO QUEIMADO,pac_CHAPECO
0,2018-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
1,2018-01-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2,2018-01-03,18.2,36.8,32.6,91.2,0.0,28.8,34.6,14.2,33.4,7.6,36.4,21.0,21.8,NaN
3,2018-01-04,18.2,17.6,19.4,22.2,0.0,17.8,17.0,4.8,30.6,7.0,36.4,20.2,21.2,NaN
4,2018-01-05,16.2,0.6,7.6,0.4,0.0,0.2,0.2,0.4,0.8,4.4,0.0,5.4,4.2,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2917,2025-12-27,0.0,0.0,13.8,0.0,0.0,0.0,0.0,0.0,1.0,9.2,21.6,2.6,20.0,4.6
2918,2025-12-28,0.0,0.0,9.4,0.0,0.0,0.0,0.0,0.0,0.6,0.2,38.6,0.6,47.4,19.6
2919,2025-12-29,0.0,0.0,30.2,0.0,0.0,0.0,0.0,0.0,66.6,22.4,54.8,0.4,77.4,49.6
2920,2025-12-30,0.0,0.0,32.0,0.0,0.0,0.0,0.0,0.0,90.4,64.2,37.8,0.4,76.8,52.6


## Resto ------------------------------------------

Tarefas

In [ ]:
#scatter severidade diaria plot
x_inicio = fln['Data Medicao'].min()
x_fim = fln['Data Medicao'].max()
contagem = fln['severidade'].value_counts()
texto_legenda = "<b>Total de Ocorrências:</b><br>" + "<br>".join([f"{cat}: {val}" for cat, val in contagem.items()])

fig = px.scatter(
    fln, 
    x='Data Medicao', 
    y='PRECIPITACAO TOTAL, HORARIO(mm)', 
    color="severidade",
    color_discrete_map={
        "Sem Chuva": "gray", "Leve": "blue", "Moderada": "green", "Intensa": "orange", "Extrema": "red"
    },
    category_orders={"sev": ordem_chuva},
    title="Análise de Chuva em Florianópolis",
    labels={"PRECIPITACAO TOTAL, HORARIO(mm)": "Chuva (mm)", "Data Medicao": "Data"},
    hover_data=['PRECIPITACAO TOTAL, HORARIO(mm)']
)

fig.add_annotation(
    dict(
        xref="paper", yref="paper",
        x=1.115, y=0.7, # Posiciona à direita, no topo
        text=texto_legenda,
        showarrow=False,
        align="left",
        bgcolor="rgba(255, 255, 255, 0.8)", # Fundo branco semi-transparente
        bordercolor="black",
        borderwidth=1,
        font=dict(size=14)
    )
)

# Aumentar pontos e fontes (como feito antes)
fig.update_traces(marker=dict(size=9))
fig.update_layout(font=dict(size=20), title_font=dict(size=24))

# 2. Adicionar as linhas apenas no domínio dos dados usando shapes
# Linha Moderada (y=10)
fig.add_shape(type="line", x0=x_inicio, x1=x_fim, y0=10, y1=10,
              line=dict(color="green", width=1, dash="dot"))

# Linha Intensa (y=50)
fig.add_shape(type="line", x0=x_inicio, x1=x_fim, y0=50, y1=50,
              line=dict(color="orange", width=1, dash="dot"))


# Linha Extrema (y=100)
fig.add_shape(type="line", x0=x_inicio, x1=x_fim, y0=100, y1=100,
              line=dict(color="red", width=1, dash="dot"))


fig.show(renderer="browser")


In [ ]:
#scatter plot ACUMULADO 3 DIAS
x_inicio = fln['Data Medicao'].min()
x_fim = fln['Data Medicao'].max()
contagem = fln['sev_acumulada'].value_counts()
texto_legenda = "<b>Total de Ocorrências:</b><br>" + "<br>".join([f"{cat}: {val}" for cat, val in contagem.items()])

fig = px.scatter(
    fln, 
    x='Data Medicao', 
    y='acumulado_3dias', 
    color="sev_acumulada",
    color_discrete_map={
        "Sem Chuva": "gray", "Leve": "blue", "Moderada": "green", "Intensa": "orange", "Extrema": "red"
    },
    category_orders={"sev_acumulada": ordem_chuva},
    title="Análise de Chuva Acumulada em Florianópolis",
    labels={"acumulado_3dias": "Chuva Acumulada (mm)", "Data Medicao": "Data"},
    hover_data=['acumulado_3dias']
)

fig.add_annotation(
    dict(
        xref="paper", yref="paper",
        x=1.115, y=0.7, # Posiciona à direita, no topo
        text=texto_legenda,
        showarrow=False,
        align="left",
        bgcolor="rgba(255, 255, 255, 0.8)", # Fundo branco semi-transparente
        bordercolor="black",
        borderwidth=1,
        font=dict(size=14)
    )
)

# Aumentar pontos e fontes (como feito antes)
fig.update_traces(marker=dict(size=9))
fig.update_layout(font=dict(size=20), title_font=dict(size=24))

# 2. Adicionar as linhas apenas no domínio dos dados usando shapes
# Linha Moderada (y=10)
fig.add_shape(type="line", x0=x_inicio, x1=x_fim, y0=10, y1=10,
              line=dict(color="green", width=1, dash="dot"))

# Linha Intensa (y=50)
fig.add_shape(type="line", x0=x_inicio, x1=x_fim, y0=50, y1=50,
              line=dict(color="orange", width=1, dash="dot"))


# Linha Extrema (y=100)
fig.add_shape(type="line", x0=x_inicio, x1=x_fim, y0=100, y1=100,
              line=dict(color="red", width=1, dash="dot"))


fig.show(renderer="browser")

In [83]:
len(dbs['FLORIANOPOLIS']['df_days'])

2193